# Evaluating a planner, step by step

`passage-eval` scores any planner that turns a PASSAGE task into a list of cells. This notebook shows
the whole contract on a deliberately naive planner. It writes a planner in a few lines, runs it on one
task next to the exact A\* planner, reads each of the six primary metrics on that task, and then scores
both planners on a small sample.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It solves a few dozen tasks; about half a minute.

In [ ]:
resolution = 256
split = "validation"
objective = "elevation"  # elevation, slope_uphill or energy
condition = "obstacles"  # obstacles or free
tasks = 16  # validation tasks of Section 4
output_dir = "notebook-artifacts/notebooks/passage-eval/01_astar_walkthrough"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "65f1e4f0a37d8a7b476ddf02d48fe542542924ea"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))


# The shared trajectory style of the paper: one terrain scale, white start circle and goal star,
# green solid routes on free terrain and blue short dashes when obstacles are blocked.
from trajectory_style import render_cost_comparison, draw_comparison, comparison_handles  # noqa: E402, F401
from passage.plotting import decode_scene, render_scene, terrain_colormap, OBSTACLE_COLOR  # noqa: E402, F401


def first_scene(resolution, split, min_share=0.25, min_coverage=0.05):
    """Selection rule fixed in advance: the first scene of the split whose six stored routes all span at least
    `min_share` of the grid side and whose obstacles cover at least `min_coverage` of the cells."""
    for candidate in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False):
        lengths = [variant["length"] for paths in candidate["metadata"]["paths"].values()
                   for variant in (paths["free"], paths["obstacles"])]
        coverage = candidate["metadata"]["obstacles"]["ratio_actual"]
        if min(lengths) >= min_share * resolution and coverage >= min_coverage:
            return candidate["id"]
    raise ValueError("No scene of the split meets the selection rule")


from passage_eval.baselines.classical import AStarPlanner, GreedyToGoalPlanner, RandomPolicyPlanner  # noqa: E402, F401
from passage_eval.io.dataset import scene_from_row  # noqa: E402
from passage_eval.protocol.planner import Planner, PlannerOutput, evaluate_planner  # noqa: E402, F401

METRICS = ["cost_regret_median", "cost_regret_p90", "path_assd", "path_hausdorff_p95",
           "success_rate", "obstacle_violation_rate"]
PLANNER_COLORS = {"A*": "#238b45", "Greedy": "#1767ac", "Random walk": "#b8862b", "Straight line": "#1767ac"}


SOLVER_OF = {"elevation": "astar_elevation", "slope_uphill": "astar_slope", "energy": "astar_energy"}


def load_tasks(resolution, objective, condition, limit):
    """First `limit` scored tasks of the split; proved no-path variants have no reference route and are skipped."""
    columns = ["id", "tensor", "metadata"] + [f"path_astar_{s}_{condition}" for s in ("elevation", "slope", "energy")]
    tasks = []
    for row in source.iter_rows(resolution, split, columns=columns, samples=False):
        if row["metadata"]["paths"][SOLVER_OF[objective]][condition]["status"] != "ok":
            continue
        tasks.append(scene_from_row(row, resolution=resolution, cost_model=objective, split=split, task_variant=condition))
        if len(tasks) == limit:
            return tasks
    return tasks

## 1. The planner contract

A planner subclasses `Planner` and implements `predict_path(scene)`. The scene it receives holds the
elevation in metres, the obstacle mask, the start and the goal; the evaluator removes the reference
route and its cost before the call. The planner returns the ordered cells of its route. The cost it
reports is ignored: the evaluator recomputes it.

The straight-line planner below draws the line from start to goal and ignores both the terrain and the
obstacles. It is a useful foil: its routes always connect, but they may cross obstacles and climb hills.

In [ ]:
from dataclasses import dataclass


@dataclass
class StraightLinePlanner(Planner):
    """Cells on the straight segment from start to goal; ignores costs and obstacles."""

    name: str = "straight_line"

    def predict_path(self, scene):
        (r0, c0), (r1, c1) = scene.start, scene.goal
        steps = max(abs(r1 - r0), abs(c1 - c0))
        rows = np.rint(np.linspace(r0, r1, steps + 1)).astype(int)
        cols = np.rint(np.linspace(c0, c1, steps + 1)).astype(int)
        return PlannerOutput(predicted_path=list(zip(rows.tolist(), cols.tolist())), predicted_cost=float("nan"))

## 2. One task, two planners

Figure 1 draws both routes against the reference in the paper's comparison style: the reference in
solid green, the planner's route in dashed blue.

In [ ]:
task = load_tasks(resolution, objective, condition, 1)[0]
planners = {"A*": AStarPlanner(), "Straight line": StraightLinePlanner()}
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.8))
fig.subplots_adjust(top=0.82, bottom=0.14, wspace=0.06)
one_task = []
for ax, (name, planner) in zip(axes, planners.items()):
    route = planner.predict_path(task).predicted_path
    draw_comparison(task.elevation, task.obstacles, task.start, task.goal, task.oracle_path, route, ax=ax)
    ax.set_title(name, fontsize=10.5)
    metrics, _ = evaluate_planner(planner, [task])
    one_task.append({"Planner": name, **{m: metrics[m] for m in METRICS}})
fig.legend(handles=comparison_handles(), loc="lower center", ncol=5, fontsize=8.5)
figure_title(fig, f"Task {task.sample_id}: reference and candidate routes",
             f"{resolution}×{resolution} · {OBJECTIVE_LABELS[objective]} objective · {condition}")
figures = save_figure(fig, output_path, f"01_astar_walkthrough-{resolution}-{task.sample_id}")
plt.show()
one_task = pd.DataFrame(one_task)
show(one_task, f"Table 1. The six metrics on task {task.sample_id}.", "{:,.4f}")

*Figure 1. Reference route (solid green) and planner route (dashed blue) for the exact A\* planner
(left) and the straight-line planner (right). Hatched cells are obstacles.*

## 3. Reading the six metrics

- **Cost regret** (`cost_regret_median`, `cost_regret_p90`): the planner's cost minus the reference
  cost, divided by the reference cost. A route that crosses an obstacle or fails to connect receives a
  censored cost of a million times the reference, hence a regret of 999,999.
- **Geometry** (`path_assd`, `path_hausdorff_p95`): distances between the planner's cells and the
  reference cells, computed only on valid routes. ASSD is normalised by the reference length; H95 is in
  cells. A tied optimum can have zero regret and non-zero geometry.
- **Reliability** (`success_rate`, `obstacle_violation_rate`): the share of tasks whose route connects
  start and goal, and the share whose route enters an obstacle cell.

In [ ]:
line = one_task.set_index("Planner").loc["Straight line"]
verdict = ("crosses an obstacle, so its cost is censored and its geometry is not computed"
           if line.obstacle_violation_rate > 0 else
           f"stays clear of the obstacles and costs {100 * line.cost_regret_median:.1f} % more than the optimum")
display(Markdown(f"On this task, the straight line {verdict}. A\\* reproduces the reference with zero regret."))

## 4. A small sample

One task proves little. Table 2 scores both planners on the first `tasks` validation tasks.

In [ ]:
sample = load_tasks(resolution, objective, condition, tasks)
rows = []
for name, planner in planners.items():
    metrics, _ = evaluate_planner(planner, sample)
    rows.append({"Planner": name, **{m: metrics[m] for m in METRICS}})
scores = pd.DataFrame(rows)
show(scores, f"Table 2. Six primary metrics on {len(sample)} validation tasks.", "{:,.4f}")

## Summary

In [ ]:
s = scores.set_index("Planner")
display(Markdown("\n".join([
    "- Any object with a `predict_path` method that returns cells can be scored; the evaluator recomputes "
    "every cost itself.",
    f"- On {len(sample)} tasks, the straight line connects {100 * s.loc['Straight line', 'success_rate']:.0f} % of "
    f"them but enters an obstacle in {100 * s.loc['Straight line', 'obstacle_violation_rate']:.0f} %, which "
    "dominates its regret.",
    "- Report the six metrics together: regret, geometry and reliability each catch failures the others miss.",
])))
report = {"resolution": resolution, "split": split, "objective": objective, "condition": condition,
          "task": task.sample_id, "one_task": one_task.to_dict("records"), "sample": scores.to_dict("records"),
          "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"01_astar_walkthrough-{resolution}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*